# RP1: imitación de movimientos en Genesis-Forge (GPU)

Entrena una sola política para los 9 movimientos de `config/motions.json` con miles de entornos en paralelo y la exporta a TorchScript (`rp1_policy.pt`) para ejecutarla en el entorno MuJoCo local con el agente de lenguaje.

**Entorno de ejecución → Cambiar tipo de entorno → GPU** antes de empezar.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
# El submódulo trae el modelo del robot (third_party/RP1_description)
!git clone --recurse-submodules https://github.com/SirJuanGod/Agent_IA_HKTN.git
%cd Agent_IA_HKTN

In [ ]:
# genesis-forge pide torch>=2.10: si pip actualiza torch, reinicia el entorno (Entorno de ejecución → Reiniciar) y sigue desde la celda siguiente
!pip install -q genesis-forge rsl-rl-lib tensorboard mujoco gymnasium
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Checkpoints en Drive: Colab se desconecta y así no se pierde el entrenamiento
from google.colab import drive
drive.mount('/content/drive')
LOG_DIR = '/content/drive/MyDrive/rp1_genesis'

In [ ]:
# Paridad Genesis ↔ MuJoCo (CPU, ~1 min): la observación al arrancar debe coincidir (altura: 0.002 de margen)
%cd /content/Agent_IA_HKTN
!python experiments/test_rp1_genesis.py --motion squat_front_raise --frame 60 --steps 30

In [ ]:
# Prueba corta para medir velocidad (pasos/s) y memoria; si da OOM baja --envs a 2048
!python experiments/train_rp1_genesis.py --envs 4096 --iterations 5 --log-dir /content/rp1_bench

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR

In [ ]:
# Entrenamiento completo (~3e8 pasos). Guarda model_<it>.pt cada 100 iteraciones y al final rp1_policy.pt
!python experiments/train_rp1_genesis.py --envs 4096 --iterations 3000 --log-dir $LOG_DIR

In [ ]:
# Si Colab se desconectó: reanuda desde el último checkpoint
import glob, re
last = max(glob.glob(f'{LOG_DIR}/model_*.pt'), key=lambda p: int(re.findall(r'model_(\d+)', p)[0]))
print(last)
!python experiments/train_rp1_genesis.py --envs 4096 --iterations 1000 --log-dir $LOG_DIR --resume "$last"

## Usarla en local

Descarga `rp1_policy.pt` de Drive (`MyDrive/rp1_genesis/`) a `checkpoints/rp1_genesis_policy.pt` y ejecuta:

```bash
python experiments/run_rp1_language.py --phase floating --policy checkpoints/rp1_genesis_policy.pt
python experiments/play_rp1_motion.py --phase floating --motion forward_lunge --policy checkpoints/rp1_genesis_policy.pt
```